# 24｜葡萄牙污水冻结模型推理与身份盲态危险性排序

本笔记本读取23号冻结的79,867个代表性DDA-MS/MS特征，使用19号已经选定的模型、校准和拒绝阈值进行一次性真实水样推理。这里不读取名称、结构、分子式、谱库命中或原论文注释，也不重新训练或重新选择模型。

流程分三段并支持断点复用：`env_ms`冻结模型输入 → `env_dreams`提取DreaMS表示 → `env_ms`运行8条候选管线、计算冻结不确定性并生成身份盲态排序。只有最终五个端点各自预选的唯一模型产生主预测；其余候选管线仅用于复现19号冻结的模型分歧不确定性。

In [ ]:
from pathlib import Path
import json, os, subprocess, sys
import pandas as pd
from IPython.display import display, Image

ROOT = Path(r'E:\谱图预测PBT_终版')
ENV_MS = Path(r'D:\Appinstallation\Anaconda\envs\env_ms\python.exe')
ENV_DREAMS = Path(r'D:\Appinstallation\Anaconda\envs\env_dreams\python.exe')
SRC = ROOT / 'src'
CASE = ROOT / 'external_data/portugal_wastewater_5830725'
MANIFEST_DIR = CASE / 'manifests'
OUT = CASE / '04_hazard_ranking_24'

# Canonical-score rebuild: reuse the already frozen inputs and embeddings.
# The predict stage reuses credential-checked endpoint prediction caches and rewrites the ranking outputs.
RUN_PREPARE = False
RUN_DREAMS_EMBED = False
RUN_FROZEN_INFERENCE = True
DREAMS_BATCH_SIZE = 16
INFERENCE_BATCH_SIZE = 128

assert ENV_MS.is_file(), ENV_MS
assert ENV_DREAMS.is_file(), ENV_DREAMS
env = os.environ.copy()
env['PYTHONPATH'] = str(SRC) + os.pathsep + env.get('PYTHONPATH', '')
env['PYTHONIOENCODING'] = 'utf-8'
print('env_ms:', ENV_MS)
print('env_dreams:', ENV_DREAMS)
print('output:', OUT)

## 1. 23号门控与身份盲态约束

必须显示 `READY_FOR_NOTEBOOK_24`、96/96进样完成和全部必需QC通过。24号禁止读取任何身份字段；排序冻结后，25号才能进行谱库/原论文注释匹配。

In [ ]:
nb23_path = MANIFEST_DIR / '23_feature_extraction_run_manifest.json'
nb23 = json.loads(nb23_path.read_text(encoding='utf-8'))
assert nb23['status'] == 'READY_FOR_NOTEBOOK_24', nb23['status']
assert nb23['qc_all_required_pass'] is True
assert nb23['counts']['runs'] == 96
assert nb23['counts']['sample_units'] == 48
assert nb23['counts']['sample_consensus_model_eligible_ms2_features'] == 79867
display(pd.Series({
    'status': nb23['status'],
    'runs': nb23['counts']['runs'],
    'sample_units': nb23['counts']['sample_units'],
    'eligible_features': nb23['counts']['sample_consensus_model_eligible_ms2_features'],
    'blank_limitation': nb23['blank_limitation'],
}, name='value').to_frame())

## 2. 冻结79,867个模型输入（env_ms）

生成Binned MS/MS、Transformer token和DreaMS MGF，并在正负模式内按10 ppm、15 s进行不读取身份的跨样品特征分组。`PS-F_C3`等低MS2深度样品会自动保留并显式标记，不会静默删除。首次运行通常需要数分钟。

In [ ]:
prepare_manifest_path = MANIFEST_DIR / '24_preparation_manifest.json'
if RUN_PREPARE or not prepare_manifest_path.exists():
    cmd = [str(ENV_MS), '-m', 'pbt_msms.portugal_hazard_ranking',
           '--stage', 'prepare', '--root', str(ROOT)]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=ROOT, env=env)
prepare = json.loads(prepare_manifest_path.read_text(encoding='utf-8'))
assert prepare['status'] == 'PASS'
assert prepare['identity_blind'] is True
assert prepare['n_features'] == 79867
display(pd.Series({k: prepare[k] for k in [
    'status', 'n_features', 'n_samples', 'n_case_feature_groups',
    'mode_counts', 'low_ms2_depth_samples', 'identity_blind'
]}, name='value').to_frame())

## 3. DreaMS冻结表示（env_dreams，GPU）

为P筛查最终模型生成79,867×1,024维DreaMS表示。该步骤只读取已冻结MGF。默认batch size 16与此前流程一致；若显存不足，只降低 `DREAMS_BATCH_SIZE`，不要改变谱图或模型。

In [ ]:
embed_manifest_path = MANIFEST_DIR / '24_dreams_embedding_manifest.json'
if RUN_DREAMS_EMBED and not embed_manifest_path.exists():
    cmd = [str(ENV_DREAMS), '-m', 'pbt_msms.portugal_hazard_ranking',
           '--stage', 'embed', '--root', str(ROOT),
           '--batch-size', str(DREAMS_BATCH_SIZE)]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=ROOT, env=env)
embed = json.loads(embed_manifest_path.read_text(encoding='utf-8'))
assert embed['status'] == 'PASS'
assert embed['shape'] == [79867, 1024]
display(pd.Series({k: embed[k] for k in ['status', 'shape', 'dtype', 'batch_size', 'gpu']}, name='value').to_frame())

## 4. 冻结模型推理、不确定性与身份盲态排序（env_ms，GPU）

此阶段计算8条候选管线的部署集成均值，以严格复现19号的 `direct_candidate_model_disagreement`；Transformer B的B/NOEC不确定性仍使用五折内三种子标准差。完整运行会依次读取约80个深度模型checkpoint，是24号最慢的部分，常见耗时约数小时，取决于GPU和硬盘。每条候选管线完成后单独落盘，意外中断可直接重跑本单元继续。

主排序使用Development OOF锚定的P、B和T危险性百分位：T为EC50、LC50、NOEC三个反向百分位的中位数，P/B/T三部分取几何平均；只有五个端点均通过冻结拒绝阈值的特征进入主排序。检出频率辅助排序单独保存，不与危险性主排名混为一谈。

In [ ]:
run_manifest_path = MANIFEST_DIR / '24_hazard_ranking_run_manifest.json'
if RUN_FROZEN_INFERENCE or not run_manifest_path.exists():
    cmd = [str(ENV_MS), '-m', 'pbt_msms.portugal_hazard_ranking',
           '--stage', 'predict', '--root', str(ROOT),
           '--batch-size', str(INFERENCE_BATCH_SIZE)]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=ROOT, env=env)
run = json.loads(run_manifest_path.read_text(encoding='utf-8'))
qc = pd.read_csv(MANIFEST_DIR / '24_hazard_ranking_qc_checks.csv')
display(qc)
assert qc['passed'].astype(bool).all(), qc.loc[~qc['passed'].astype(bool)]
assert run['status'] == 'READY_FOR_NOTEBOOK_25_IDENTITY_REVEAL', run['status']
display(pd.Series({
    'status': run['status'],
    'device': run['device'],
    'external_recalibration': run['external_recalibration'],
    'identity_access': run['identity_access'],
    'selected_models': run['selected_models'],
}, name='value').to_frame())

## 5. 完成状态与输出

只有全部QC通过并显示 `READY_FOR_NOTEBOOK_25_IDENTITY_REVEAL` 才算24号完成。此时排名和文件哈希已经冻结，之后25号才能匹配原论文注释、谱库和已知污染物，并计算Top 5%/10%/20%的召回、precision和富集倍数。

当前结果仍称为**危险性优先排序**。检出频率只是出现信息，不等于浓度或暴露；公开数据缺少可识别空白，因此也不能将全部特征解释为确认污染物。

In [ ]:
freeze = json.loads((MANIFEST_DIR / '24_identity_blind_freeze.json').read_text(encoding='utf-8'))
display(pd.Series({
    'freeze_status': freeze['status'],
    'n_features': freeze['n_features'],
    'n_primary_ranked': freeze['n_primary_ranked'],
    'n_case_feature_groups': freeze['n_case_feature_groups'],
    'low_ms2_depth_samples': freeze['low_ms2_depth_samples'],
    'blank_limitation': freeze['blank_limitation'],
}, name='value').to_frame())
display(Image(filename=str(OUT / 'figures/24_identity_blind_overview.png')))
print('逐特征冻结预测:', OUT / '24_feature_predictions_identity_blind.csv.gz')
print('跨样品特征组:', OUT / '24_case_feature_groups_identity_blind.csv.gz')
print('样品汇总:', OUT / '24_sample_priority_summary.csv')
print('站点×批次汇总:', OUT / '24_site_campaign_priority_summary.csv')
print('身份盲态冻结记录:', MANIFEST_DIR / '24_identity_blind_freeze.json')
print('STATUS:', run['status'])